# Shard collection: Distributed probe data collection

Generates reasoning traces and extracts hidden states for a **subset** of AIME problems, producing one `shard_<NAME>.pt` file. 

Each person on a disjoint problem range, in parallel. The resulting shards are later merged by `shard_merge.ipynb` and used to train the probe in `probe_train.ipynb`.

## Workflow at a glance

```
[Person 1] shard_collect.ipynb (problems   0-249) → shard_1.pt  ─┐
[Person 2] shard_collect.ipynb (problems 250-499) → shard_2.pt  ─┤
[Person 3] shard_collect.ipynb (problems 500-749) → shard_3.pt  ─┼ → shard_merge.ipynb → X.pt, y.pt, problem_ids.pt
[Person 4] shard_collect.ipynb (problems 750-999) → shard_4.pt  ─┘                      ↓
                                                                                     probe_train.ipynb → linear_probe.pt + metrics
```

## Suggested problem splits

*No two people should have overlap*

| # People | Person | START_IDX | END_IDX |
|---|---|---|---|
| 2 | A | 0 | 500 |
|   | B | 500 | 1000 |
| 3 | A | 0 | 334 |
|   | B | 334 | 667 |
|   | C | 667 | 1000 |
| 4 | A | 0 | 250 |
|   | B | 250 | 500 |
|   | C | 500 | 750 |
|   | D | 750 | 1000 |

---

> **Note:** AIME problems are ordered by year, so later ranges (e.g. 750–1000) contain harder problems with longer traces. The person taking the latest range should expect ~1.3–1.5× higher unit cost than the one taking the earliest range.

---
## What this saves

`shard_<NAME>.pt` -> A single torch pickle with:
- `X`: `[total_positions, 3584]` `float32`: hidden states at paragraph breaks in the CoT
- `y`: `[total_positions]` `float32`: per-trace correctness labels (1 if trace's final answer was correct)
- `problem_ids`: `[total_positions]` `int64`: which problem each position came from (needed for problem-level train/test split)
- `meta`: dict with `NAME`, `START_IDX`, `END_IDX`, `MODEL_ID`, `MAX_NEW_TOKENS`, etc.

Also writes a human-readable `shard_<NAME>_results.csv` with per-problem correctness info for debugging.

## Before you run

1. **Check which range you're gonna work on** Choose the correct range
2. **Choose a unique `NAME`** (it'll be in the filename)
3. **Set the GPU runtime:** Runtime → Change runtime type → H100 (or A100). T4 is not recommended for the 7B model at full `MAX_NEW_TOKENS`
4. **Check your compute units** before starting. A 250-problem shard runs ~5 hr on H100 (~50 units at 8.71 units/hr); budget accordingly
5. **Do not modify anything but the config cell below** Otherwise your shard won't be compatible with the others

In [ ]:
# === Colab bootstrap ===
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/Avni2000/cs639-final-project.git"
    REPO_DIR = "/content/cs639-final-project"
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull

    %cd {REPO_DIR}/probe-baseline

    OUTPUT_DIR = "/content/drive/MyDrive/cs639-outputs"
else:
    OUTPUT_DIR = "."

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"IN_COLAB={IN_COLAB}  OUTPUT_DIR={OUTPUT_DIR}")

In [ ]:
%pip install -q "transformers==4.45.0" "torch>=2.3" "datasets" "pandas" \
               "math-verify[antlr4_13_2]" "tqdm"

## Config

In [ ]:
# ========== YOUR SETTINGS ==========
NAME        = "CHANGE_ME"    # your first name or unique id; e.g. "avni"
START_IDX   = 0              # inclusive; see split table in the intro markdown
END_IDX     = 250            # exclusive
# ====================================

# ========== SHARED CONFIG ==========
MODEL_ID               = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
HIDDEN_DIM             = 3584     # DeepSeek-R1-Distill-Qwen-7B
PROBE_LAYER            = -1       # last hidden layer (= layer 28)
MAX_NEW_TOKENS         = 16384    # cap on CoT length
MAX_HIDDEN_PER_PROBLEM = 100      # uniform-random subsample of paragraph breaks
PARAGRAPH_SEED         = 0        # RNG seed for paragraph subsampling
CHECKPOINT_EVERY       = 25       # save a partial shard every N problems
CSV_PATH               = "aime_historical.csv"
# =============================================================================

assert NAME != "CHANGE_ME", "Set NAME to your identifier before running."
assert 0 <= START_IDX < END_IDX <= 1000, "Invalid problem range."
print(f"{NAME}: problems {START_IDX}..{END_IDX - 1}  ({END_IDX - START_IDX} problems)")

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU required. Runtime -> Change runtime type -> GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# === Download the AIME dataset if missing, load it ===
import pandas as pd

if not os.path.exists(CSV_PATH):
    from datasets import load_dataset
    ds = load_dataset("gneubig/aime-1983-2024")
    ds["train"].to_csv(CSV_PATH)

df = pd.read_csv(CSV_PATH)
problems = df[["Question", "Answer"]].dropna().reset_index(drop=True)
assert END_IDX <= len(problems), f"END_IDX={END_IDX} exceeds dataset size {len(problems)}"
subset = problems.iloc[START_IDX:END_IDX]
print(f"Dataset size: {len(problems)} problems | your range: {len(subset)} problems")

In [ ]:
# === Load the model (~1 min download on first run, cached in /content after) ===
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")

In [ ]:
# === Helpers (same logic as train.ipynb) ===
import random

def build_prompt(problem_text: str) -> str:
    messages = [{
        "role": "user",
        "content": (
            "Solve the following AIME problem. "
            "Your final answer must be a non-negative integer (0-999), "
            "placed inside \\boxed{}.\n\n" + problem_text
        ),
    }]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _find_subseq(seq, sub):
    n, m = len(seq), len(sub)
    for i in range(n - m + 1):
        if seq[i:i + m] == sub:
            return i
    return -1

def _paragraph_break_indices(cot_token_ids):
    """Indices within cot_token_ids whose decode completes a \\n\\n break."""
    positions = []
    cursor = 0
    for i in range(len(cot_token_ids)):
        text = tokenizer.decode(cot_token_ids[:i + 1], skip_special_tokens=False)
        hit = text.find("\n\n", cursor)
        if hit != -1:
            positions.append(i)
            cursor = hit + 2
    return positions

def get_cot_hidden_states(prompt, rng=None):
    """Returns (hidden_states [N_pos, HIDDEN_DIM], generated_text)."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        gen = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
            output_hidden_states=True,
        )

    generated_ids = gen.sequences[0, prompt_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=False)
    gen_list = generated_ids.tolist()

    if gen.hidden_states:
        hs = torch.cat(
            [step[PROBE_LAYER][:, -1, :] for step in gen.hidden_states],
            dim=0,
        ).float().cpu()
    else:
        hs = torch.empty((0, HIDDEN_DIM), dtype=torch.float32)

    think_toks = tokenizer.encode("<think>",  add_special_tokens=False)
    end_toks   = tokenizer.encode("</think>", add_special_tokens=False)
    tp = _find_subseq(gen_list, think_toks)
    ep = _find_subseq(gen_list, end_toks)
    if tp != -1 and ep != -1 and ep > tp:
        cot_start, cot_end = tp + len(think_toks), ep
    else:
        cot_start, cot_end = 0, len(gen_list)

    cot_ids = gen_list[cot_start:cot_end]
    rel = _paragraph_break_indices(cot_ids)
    if len(rel) > MAX_HIDDEN_PER_PROBLEM:
        r = rng if rng is not None else random
        rel = sorted(r.sample(rel, MAX_HIDDEN_PER_PROBLEM))

    if rel:
        abs_pos = torch.tensor([cot_start + p for p in rel], dtype=torch.long)
        return hs.index_select(0, abs_pos), generated_text
    return torch.empty((0, HIDDEN_DIM), dtype=torch.float32), generated_text

In [ ]:
# === Collection loop with periodic checkpointing ===
from math_verify import parse, verify
from tqdm.auto import tqdm

rng = random.Random(PARAGRAPH_SEED)
shard_path   = os.path.join(OUTPUT_DIR, f"shard_{NAME}.pt")
partial_path = os.path.join(OUTPUT_DIR, f"shard_{NAME}_partial.pt")
results_path = os.path.join(OUTPUT_DIR, f"shard_{NAME}_results.csv")

X_list, y_list, problem_ids = [], [], []
results = []
t_start = time.time()

for pos, (idx, row) in enumerate(tqdm(subset.iterrows(), total=len(subset),
                                      desc=f"{NAME} collecting")):
    prompt = build_prompt(row["Question"])
    t0 = time.time()
    cot_hidden, gen_text = get_cot_hidden_states(prompt, rng=rng)
    wall = time.time() - t0

    gold = parse(f"${row['Answer']}$")
    pred = parse(gen_text)
    is_correct = bool(verify(gold, pred)) if pred else False
    label = 1 if is_correct else 0
    pred_str = str(pred[0]) if pred else "<none>"

    if cot_hidden.shape[0] > 0:
        X_list.append(cot_hidden)
        y_list.extend([label] * cot_hidden.shape[0])
        problem_ids.extend([int(idx)] * cot_hidden.shape[0])

    results.append({
        "problem_idx": int(idx),
        "correct_answer": str(row["Answer"]),
        "predicted": pred_str,
        "label": label,
        "positions": cot_hidden.shape[0],
        "wall_s": wall,
    })
    print(f"  [{pos+1}/{len(subset)}] problem_idx={idx} label={label} "
          f"positions={cot_hidden.shape[0]} wall={wall:.1f}s")

    # checkpoint
    if (pos + 1) % CHECKPOINT_EVERY == 0:
        torch.save({
            "X": torch.cat(X_list, dim=0) if X_list else torch.empty(0, HIDDEN_DIM),
            "y": torch.tensor(y_list, dtype=torch.float32),
            "problem_ids": torch.tensor(problem_ids, dtype=torch.long),
            "progress": pos + 1,
        }, partial_path)
        pd.DataFrame(results).to_csv(results_path, index=False)
        print(f"  [checkpoint] saved partial to {partial_path}")

print(f"\nTotal wall time: {(time.time() - t_start) / 60:.1f} min")
print(f"Correct / total: {sum(r['label'] for r in results)} / {len(results)}")

In [ ]:
# === Save the final shard ===
X = torch.cat(X_list, dim=0) if X_list else torch.empty(0, HIDDEN_DIM)
y = torch.tensor(y_list, dtype=torch.float32)
pids = torch.tensor(problem_ids, dtype=torch.long)

shard = {
    "X": X,
    "y": y,
    "problem_ids": pids,
    "meta": {
        "NAME": NAME,
        "START_IDX": START_IDX,
        "END_IDX": END_IDX,
        "MODEL_ID": MODEL_ID,
        "PROBE_LAYER": PROBE_LAYER,
        "HIDDEN_DIM": HIDDEN_DIM,
        "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
        "MAX_HIDDEN_PER_PROBLEM": MAX_HIDDEN_PER_PROBLEM,
        "PARAGRAPH_SEED": PARAGRAPH_SEED,
    },
}
torch.save(shard, shard_path)
pd.DataFrame(results).to_csv(results_path, index=False)

# Clean up partial checkpoint
if os.path.exists(partial_path):
    os.remove(partial_path)

print(f"\n✓ Shard saved    -> {shard_path}")
print(f"✓ Results saved  -> {results_path}")
print(f"\nShard stats:")
print(f"  X: {X.shape}  y: {y.shape}  problem_ids: {pids.shape}")
print(f"  labels: {int(y.sum())} correct / {int((1 - y).sum())} incorrect")

## Next steps

1. **Locate your shard file:** `MyDrive/cs639-outputs/shard_<NAME>.pt`
2. **Share it with the merge-runner:** Upload your shard to shared Drive folder.
3. **Upload your notebook in cs639-final-project/probe-baseline/shards-logs** To analyse later what your label distribution looked like (Stats will be printed above)

### Troubleshooting

- **OOM during generation:** Lower `MAX_NEW_TOKENS` in the config cell and coordinate with the team so everyone uses the same value.
- **Colab disconnected mid-run:** You have a `shard_<NAME>_partial.pt` checkpoint. Modify the loop to skip already-done problems using `progress` from the partial file, then rerun.